# Court Vision Mapping (CVM)

Train a YOLO26-pose model on basketball court keypoints, then use it to project on-court player positions onto a top-down court diagram.

The player detection model is trained separately (Abdo's notebook) — here we just load his weights and consume them.

## 1. Setup
Installs and imports. Run once per kernel session.

In [ ]:
%pip install torchvision==0.23
%pip install -q ultralytics supervision pyyaml
%pip install --no-cache-dir -q "git+https://github.com/roboflow/sports.git@feat/basketball"

import os
import yaml
import cv2
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm

import supervision as sv
from ultralytics import YOLO

from sports.basketball import (
    CourtConfiguration,
    League,
    draw_court,
    draw_points_on_court,
    draw_paths_on_court,
)
from sports.basketball.config import MeasurementUnit
from sports.common.view import ViewTransformer
from sports.common.team import TeamClassifier
from sports.common.path import clean_paths

SEED = 45
HOME = Path.cwd()

# Player-model class indices (verified by Zyad: ball, ball-in-basket, number, player, referee, rim).
PLAYER_CLASS_ID = 3

# Generic two-team palette for the unsupervised TeamClassifier output.
# Cluster IDs 0 and 1 are arbitrary; swap if the colours look wrong on your game.
TEAM_PALETTE = [sv.Color.from_hex("#2563eb"), sv.Color.from_hex("#ea580c")]

## 2. Dataset preparation

The court dataset is a keypoint/pose dataset — 33 vertices per image. The shipped `data.yaml` uses relative paths, so we rewrite it with absolute paths to make Ultralytics resilient to the current working directory.

In [ ]:
CVM_root = Path("Data/Datasets/basketball-court-detection-2.v1-1.yolov8/").resolve()
src_yaml_path = CVM_root / "data.yaml"

with open(src_yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg["path"] = str(CVM_root)
cfg["train"] = str(CVM_root / "train" / "images")
cfg["val"] = str(CVM_root / "valid" / "images")
cfg["test"] = str(CVM_root / "test" / "images")

updated_yaml_path = Path("cvm_data.yaml").resolve()
with open(updated_yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Wrote {updated_yaml_path}")
print(f"  train: {cfg['train']}")
print(f"  val:   {cfg['val']}")
print(f"  test:  {cfg['test']}")
print(f"  nc: {cfg['nc']}  names: {cfg['names']}  kpt_shape: {cfg.get('kpt_shape')}")

## 3. Train the CVM model

Fine-tune `yolo26n-pose.pt` on the 33-vertex court dataset. Weights auto-download on first use.

In [ ]:
CVM_model = YOLO("yolo26n-pose.pt")
CVM_model.info()

In [ ]:
CVM_results = CVM_model.train(
    data=str(updated_yaml_path),
    epochs=500,
    imgsz=640,
    batch=8,
    project="runs",
    name="CVM",
    exist_ok=True,
    deterministic=True,
    seed=SEED,
    plots=True,
    patience=30,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=4,
)

## 4. Single-frame keypoint sanity check

Load the best weights and overlay the detected court vertices on the first frame of a sample clip. Vertices below the confidence floor get zeroed out so the annotator skips them.

In [ ]:
CVM_MODEL_PATH = "runs/pose/runs/CVM/weights/best.pt"
SOURCE_VIDEO_PATH = "Data/object-tracking.mp4"

CVM_CONF = 0.3
CVM_ANCHOR_CONF = 0.5

CVM_model = YOLO(CVM_MODEL_PATH)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

result = CVM_model.predict(frame, conf=CVM_CONF, verbose=False)[0]
key_points = sv.KeyPoints.from_ultralytics(result)

# Drop low-confidence vertices by zeroing them — VertexAnnotator skips (0, 0).
confidence_mask = key_points.confidence > CVM_ANCHOR_CONF
filtered_xy = np.where(confidence_mask[..., None], key_points.xy, 0)
key_points = sv.KeyPoints(xy=filtered_xy, confidence=key_points.confidence)

vertex_annotator = sv.VertexAnnotator(color=sv.Color.RED, radius=8)
annotated_frame = vertex_annotator.annotate(scene=frame.copy(), key_points=key_points)
sv.plot_image(annotated_frame)

## 5. Fit the team classifier

Sample player crops from across the video and fit an unsupervised classifier. It learns two jersey clusters — works on any game without knowing which specific teams are playing.

In [ ]:
PLAYER_MODEL_PATH = "runs_abdo/basketball/yolo26s_960/weights/best.pt"
PLAYER_CONF = 0.25
PLAYER_IOU = 0.5
FIT_STRIDE = 30  # sample one frame every N for fitting; tune up if the video is short

PLAYER_DETECTION_MODEL = YOLO(PLAYER_MODEL_PATH)

crops = []
fit_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, stride=FIT_STRIDE)
for frame in tqdm(fit_generator, desc="collecting jersey crops"):
    result = PLAYER_DETECTION_MODEL.predict(frame, conf=PLAYER_CONF, iou=PLAYER_IOU, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    detections = detections[detections.class_id == PLAYER_CLASS_ID]
    # Shrink each box toward its centre so the crop is mostly torso (jersey), not background.
    boxes = sv.scale_boxes(xyxy=detections.xyxy, factor=0.5)
    crops.extend([sv.crop_image(frame, box) for box in boxes])

print(f"Collected {len(crops)} player crops for fitting.")

team_classifier = TeamClassifier(device="cuda" if torch.cuda.is_available() else "cpu")
team_classifier.fit(crops)
print("Team classifier fitted.")

## 6. Single-frame player → court mapping

Detect players (class 3 only), classify each by team, fit a homography from the visible court vertices, project each player's bottom-center anchor onto the top-down court, and drop anything that lands outside the court rectangle.

In [ ]:
config = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)

# Court bounds in the same units as `config` (NBA full court = 94 x 50 ft).
# Verify these match `config.length, config.width` if you change leagues.
COURT_W, COURT_H = 94, 50
COURT_MARGIN = 2

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

# --- Player detection (players only, refs/ball/rim filtered out) ---
result = PLAYER_DETECTION_MODEL.predict(frame, conf=PLAYER_CONF, iou=PLAYER_IOU, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)
detections = detections[detections.class_id == PLAYER_CLASS_ID]
detections.tracker_id = np.arange(1, len(detections.class_id) + 1)

# Predict team per player (cluster ID 0 or 1).
crops = [sv.crop_image(frame, box) for box in sv.scale_boxes(xyxy=detections.xyxy, factor=0.5)]
teams = np.array(team_classifier.predict(crops)) if crops else np.array([], dtype=int)

# Draw boxes coloured by team.
team_palette = sv.ColorPalette(TEAM_PALETTE)
box_annotator = sv.BoxAnnotator(color=team_palette, thickness=2, color_lookup=sv.ColorLookup.INDEX)
annotated_frame = box_annotator.annotate(
    scene=frame.copy(), detections=detections, custom_color_lookup=teams,
)
sv.plot_image(annotated_frame)

# --- Court keypoints + homography ---
result = CVM_model.predict(frame, conf=CVM_CONF, verbose=False)[0]
key_points = sv.KeyPoints.from_ultralytics(result)

# `key_points.confidence` is None when the CVM model finds no court at all on this frame.
if key_points.confidence is not None and len(detections) > 0:
    landmarks_mask = key_points.confidence[0] > CVM_ANCHOR_CONF
else:
    landmarks_mask = np.zeros(0, dtype=bool)

if np.count_nonzero(landmarks_mask) >= 4:
    court_landmarks = np.array(config.vertices)[landmarks_mask]
    frame_landmarks = key_points[:, landmarks_mask].xy[0]
    transformer = ViewTransformer(source=frame_landmarks, target=court_landmarks)

    frame_xy = detections.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)
    court_xy = transformer.transform_points(points=frame_xy)

    # Drop projections that land outside the court (catches benchwarmers / sideline staff).
    on_court = (
        (court_xy[:, 0] >= -COURT_MARGIN) & (court_xy[:, 0] <= COURT_W + COURT_MARGIN) &
        (court_xy[:, 1] >= -COURT_MARGIN) & (court_xy[:, 1] <= COURT_H + COURT_MARGIN)
    )
    court_xy = court_xy[on_court]
    teams_on_court = teams[on_court]

    court = draw_court(config=config)
    for team_id, colour in enumerate(TEAM_PALETTE):
        team_xy = court_xy[teams_on_court == team_id]
        if len(team_xy) > 0:
            court = draw_points_on_court(config=config, xy=team_xy, fill_color=colour, court=court)
    sv.plot_image(court)
else:
    print(f"Only {np.count_nonzero(landmarks_mask)} vertices above {CVM_ANCHOR_CONF} or no players — skipping homography.")

## 7. Full-video player → court mapping with tracking

Walk every frame, run player detection through **ByteTrack** (via Ultralytics' `model.track(persist=True)`) so each player gets a stable `tracker_id` across frames, then run the same team → homography → court pipeline as above. We accumulate per-frame `(positions, team IDs, tracker IDs)` so the cleaning step has real trajectories to operate on.

In [ ]:
# Per-frame projection with persistent tracking.
# `model.track(persist=True)` runs ByteTrack under the hood and assigns a stable
# `tracker_id` to each detection that survives across frames — that's what makes
# the cleaned trajectories meaningful.
video_xy = []
video_teams = []
video_tracker_ids = []

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

for frame in tqdm(frame_generator, desc="projecting frames"):
    # Players, tracked.
    result = PLAYER_DETECTION_MODEL.track(
        frame, persist=True, conf=PLAYER_CONF, iou=PLAYER_IOU, verbose=False
    )[0]
    detections = sv.Detections.from_ultralytics(result)
    detections = detections[detections.class_id == PLAYER_CLASS_ID]

    # Court keypoints.
    cvm_result = CVM_model.predict(frame, conf=CVM_CONF, verbose=False)[0]
    key_points = sv.KeyPoints.from_ultralytics(cvm_result)

    no_court = key_points.confidence is None
    no_players = len(detections) == 0 or detections.tracker_id is None

    if no_court or no_players:
        video_xy.append(np.zeros((0, 2)))
        video_teams.append(np.zeros((0,), dtype=int))
        video_tracker_ids.append(np.zeros((0,), dtype=int))
        continue

    landmarks_mask = key_points.confidence[0] > CVM_ANCHOR_CONF
    if np.count_nonzero(landmarks_mask) < 4:
        video_xy.append(np.zeros((0, 2)))
        video_teams.append(np.zeros((0,), dtype=int))
        video_tracker_ids.append(np.zeros((0,), dtype=int))
        continue

    # Team prediction.
    crops = [sv.crop_image(frame, box) for box in sv.scale_boxes(xyxy=detections.xyxy, factor=0.5)]
    teams = np.array(team_classifier.predict(crops))

    # Homography → court coords.
    court_landmarks = np.array(config.vertices)[landmarks_mask]
    frame_landmarks = key_points[:, landmarks_mask].xy[0]
    transformer = ViewTransformer(source=frame_landmarks, target=court_landmarks)
    frame_xy = detections.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)
    court_xy = transformer.transform_points(points=frame_xy)

    # Drop off-court projections.
    on_court = (
        (court_xy[:, 0] >= -COURT_MARGIN) & (court_xy[:, 0] <= COURT_W + COURT_MARGIN) &
        (court_xy[:, 1] >= -COURT_MARGIN) & (court_xy[:, 1] <= COURT_H + COURT_MARGIN)
    )
    video_xy.append(court_xy[on_court])
    video_teams.append(teams[on_court])
    video_tracker_ids.append(detections.tracker_id[on_court])

unique_tracks = sorted({int(t) for row in video_tracker_ids for t in row})
print(f"Projected {len(video_xy)} frames across {len(unique_tracks)} unique tracks.")

In [ ]:
# Render the raw (uncleaned) projections to an MP4, coloured by team.
source_path = Path(SOURCE_VIDEO_PATH)
TARGET_VIDEO_PATH = HOME / f"{source_path.stem}-map{source_path.suffix}"

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
sample_court = draw_court(config=config)
video_info.height, video_info.width = sample_court.shape[:2]

with sv.VideoSink(str(TARGET_VIDEO_PATH), video_info) as sink:
    for frame_xy, frame_teams in tqdm(zip(video_xy, video_teams), total=len(video_xy), desc="writing video"):
        court = draw_court(config=config)
        for team_id, colour in enumerate(TEAM_PALETTE):
            team_xy = frame_xy[frame_teams == team_id]
            if len(team_xy) > 0:
                court = draw_points_on_court(config=config, xy=team_xy, fill_color=colour, court=court)
        sink.write_frame(court)

print(f"Wrote {TARGET_VIDEO_PATH}")

## 8. Clean and smooth movement paths

Two things go wrong with raw projections:
1. **Teleports** — a misdetected court vertex flips the homography for a few frames, sending the projected point hundreds of feet across the court.
2. **Jitter** — frame-to-frame keypoint noise produces a shaky path even when the homography is roughly right.

`sports.common.path.clean_paths` (imported in setup) flags short bad runs using MAD-based speed outliers, interpolates over them, then smooths with a Savitzky–Golay filter. Because section 7 now uses ByteTrack, each `tracker_id` is a stable identity over time — we stack positions into a `(T, N, 2)` array keyed by tracker ID and `clean_paths` produces meaningful trajectories.

In [ ]:
# Build a stable (T, N, 2) array where columns are sorted unique tracker IDs.
# Now that detections are tracked, each column represents a single physical player
# across time, so cleaning + smoothing operates on real trajectories.
T = len(video_xy)
all_track_ids = sorted({int(t) for row in video_tracker_ids for t in row})
track_id_to_col = {tid: i for i, tid in enumerate(all_track_ids)}
N = len(all_track_ids)

stacked_xy = np.full((T, N, 2), np.nan, dtype=float)
stacked_teams = np.full((T, N), -1, dtype=int)
for t in range(T):
    for xy, team, tid in zip(video_xy[t], video_teams[t], video_tracker_ids[t]):
        col = track_id_to_col[int(tid)]
        stacked_xy[t, col] = xy
        stacked_teams[t, col] = int(team)

cleaned_xy, edited_mask = clean_paths(
    stacked_xy,
    jump_sigma=3.5,
    min_jump_dist=0.6,
    max_jump_run=18,
    pad_around_runs=2,
    smooth_window=9,
    smooth_poly=2,
)

# Per-track dominant team (mode), ignoring -1 padding.
def dominant_team(col):
    valid = col[col >= 0]
    if valid.size == 0:
        return 0
    vals, counts = np.unique(valid, return_counts=True)
    return int(vals[counts.argmax()])

track_team = np.array([dominant_team(stacked_teams[:, n]) for n in range(N)])

# Visualise the first track: raw (green) vs flagged segments (red).
def split_true_runs(mask, coords):
    mask = mask.squeeze()
    idx = np.flatnonzero(mask)
    if idx.size == 0:
        return []
    splits = np.where(np.diff(idx) > 1)[0] + 1
    return [coords[g] for g in np.split(idx, splits)]


raw_track = stacked_xy[:, 0, :]
court = draw_paths_on_court(
    config=config,
    paths=[raw_track[~np.isnan(raw_track).any(axis=1)]],
    color=sv.Color.GREEN,
)
court = draw_paths_on_court(
    config=config,
    paths=split_true_runs(edited_mask[:, 0], raw_track),
    color=sv.Color.RED,
    court=court,
)
sv.plot_image(court)

court = draw_paths_on_court(
    config=config,
    paths=[cleaned_xy[:, 0, :]],
    color=TEAM_PALETTE[track_team[0]],
)
sv.plot_image(court)

### Render cleaned tracks to video

In [ ]:
TARGET_CLEAN_PATH = HOME / f"{source_path.stem}-map-cleaned{source_path.suffix}"
TARGET_CLEAN_COMPRESSED_PATH = HOME / f"{TARGET_CLEAN_PATH.stem}-compressed.mp4"

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
video_info.height, video_info.width = draw_court(config=config).shape[:2]

with sv.VideoSink(str(TARGET_CLEAN_PATH), video_info) as sink:
    for t in tqdm(range(cleaned_xy.shape[0]), desc="writing cleaned video"):
        frame_xy = cleaned_xy[t]
        valid = ~np.isnan(frame_xy).any(axis=1)
        court = draw_court(config=config)
        for team_id, colour in enumerate(TEAM_PALETTE):
            team_mask = valid & (track_team == team_id)
            if team_mask.any():
                court = draw_points_on_court(
                    config=config,
                    xy=frame_xy[team_mask],
                    fill_color=colour,
                    court=court,
                )
        sink.write_frame(court)

!ffmpeg -y -loglevel error -i {TARGET_CLEAN_PATH} -vcodec libx264 -crf 28 {TARGET_CLEAN_COMPRESSED_PATH}
print(f"Wrote {TARGET_CLEAN_COMPRESSED_PATH}")